NOTE: Abhi tak tumhare experiments ke numbers CSV files aur notebook outputs mein bikhre hain (cv_ensembles_day83.csv, tuning_results_day84.csv ...). Teen mahine baad “yeh 0.78 wala run kaunse parameters, kaunse data aur kaunse code se aaya tha?” ka jawab dhoondhna mushkil hoga. Experiment tracking har run ke saath yeh sab ek jagah likh deta hai:

### MLflow ka concept

| MLflow ka concept | Matlab | Hamare project mein |
| :--- | :--- | :--- |
| **Experiment** | Runs ka group | `churn-day91-intro` |
| **Run** | Ek baar ka training/CV | `"LogReg, C=0.1"` |
| **Params** | Settings jo tum chunte ho | `hp_C`, `cv_n_splits`, data fingerprint |
| **Metrics** | Natije (step ke saath bhi) | `cv_roc_auc_mean`, fold-wise AUC |
| **Artifacts** | Files | feature list, plots |
| **Tags** | Label/context | `git_commit`, `git_dirty`, `stage` |

MLflow vs Weights & Biases (PDF dono batata hai): MLflow open-source hai, laptop par chalta hai, account nahi chahiye, aur data tumhari machine par rehta hai. W&B ke dashboards behtar hain, lekin cloud account chahiye aur data bahar jaata hai. Is project ke liye MLflow chuna hai (PDF bhi isi par focus karne ko kehta hai).

Rules:

1. Tracking mein sirf CV metrics jaayenge. Test set ka koi number log nahi hoga, taaki “kaunsa run best hai” ka faisla test dekh kar na bane (pre-registration wali baat). Isko code mein guard se enforce karenge.
2. MLflow ek development tool hai. requirements-app.txt mein nahi jaayega, aur uska store (mlflow_store/) Git mein nahi jaayega.
3. Tracking record karti hai, faisla nahi karti. Final model ka faisla abhi bhi reports/final_model_choice_day84.md mein hai.

In [10]:
import os
from pathlib import Path
import sys

# Notebook se kitni bhi directory se run ho, project root auto-detect karo.
project_root = Path.cwd().resolve()
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "data" / "processed").exists() and (candidate / "src").exists():
        project_root = candidate
        break
DATA_DIR = project_root / "data" / "processed"

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import warnings; warnings.filterwarnings('ignore')
import platform

import mlflow
import pandas as pd
import sklearn
from matplotlib.figure import Figure
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.features.build_features import NUM_V2
from src.tracking.mlflow_utils import (dataset_fingerprint, file_sha256, git_info,
                                       log_cv_metrics, setup_tracking)

print('mlflow', mlflow.__version__)
setup_tracking('churn-day91-intro')


mlflow 3.17.2


'1'

Ek “hello” run (sirf UI dekhne ke liye)

In [11]:
with mlflow.start_run(run_name='hello'):
    mlflow.log_param('purpose', 'UI dekhne ke liye')
    for step in range(5):
        log_cv_metrics({'cv_demo_curve': 1 - 0.5 ** (step + 1)}, step=step)
    mlflow.log_dict({'note': 'artifact example'}, 'note.json')

Data aur model

In [13]:
X = pd.read_csv(DATA_DIR / 'X_train_v2.csv').astype(float)
y = pd.read_csv(DATA_DIR / 'y_train.csv').squeeze()
assert len(X) == len(y)

prep = ColumnTransformer([('s', StandardScaler(), NUM_V2)], remainder='passthrough')
model = Pipeline([('prep', prep), ('m', LogisticRegression(max_iter=5000, class_weight='balanced'))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {'roc_auc': 'roc_auc', 'ap': 'average_precision', 'recall': 'recall', 'f1': 'f1'}

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\rajat\\OneDrive\\data\\processed\\X_train_v2.csv'

yeh bilkul Day 83 ka LogReg v2 setup hai, wahi folds, wahi model.

 4: Ek reusable “tracked CV run” function

In [15]:
def run_cv_experiment(run_name, model, extra_tags=None):
    est = model.named_steps['m']
    # params model se nikaale, haath se nahi likhe: warna logged params aur asli model alag ho sakte hain
    params = {'model': type(est).__name__, **{f'hp_{k}': v for k, v in est.get_params().items()}}

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({**params, 'cv_n_splits': 5, 'cv_seed': 42,
                           'scoring_primary': 'roc_auc', 'n_features': X.shape[1],
                           'data_file_sha': file_sha256(str(DATA_DIR / 'X_train_v2.csv'))[:12]})
        mlflow.log_params({f'data_{k}': v for k, v in dataset_fingerprint(X, y).items()})
        mlflow.set_tags({**git_info(), 'stage': 'cv', 'test_set_used': 'false',
                         'sklearn': sklearn.__version__, 'python': platform.python_version(),
                         **(extra_tags or {})})

        res = cross_validate(model, X, y, cv=cv, scoring=SCORING)

        # sklearn ke 'test_*' keys CV ke validation folds hain, held-out test set nahi.
        # Guard ke liye naam 'cv_' rakhte hain.
        summary = {}
        for name in SCORING:
            scores = res[f'test_{name}']
            summary[f'cv_{name}_mean'] = float(scores.mean())
            summary[f'cv_{name}_std'] = float(scores.std())
        log_cv_metrics(summary)
        for i, v in enumerate(res['test_roc_auc']):
            log_cv_metrics({'cv_fold_roc_auc': float(v)}, step=i)

        fig = Figure(figsize=(4.5, 3))
        ax = fig.subplots()
        ax.bar(range(1, 6), res['test_roc_auc'], color='steelblue')
        ax.set_ylim(0.5, 1.0)
        ax.set_xlabel('CV fold')
        ax.set_ylabel('ROC-AUC')
        ax.set_title('Fold-wise AUC (axis starts at 0.5 = chance)')
        fig.tight_layout()
        mlflow.log_figure(fig, 'fold_roc_auc.png')
        mlflow.log_dict({'features': list(X.columns)}, 'features.json')
    return run.info.run_id, summary

 5: Runs chalao

In [16]:
rid1, s1 = run_cv_experiment('logreg_v2_balanced_C1', model)
print({k: round(v, 4) for k, v in s1.items()})

# Sanity: tracked number wahi hai jo Day 83 mein report hua tha?
day83 = pd.read_csv('reports/cv_ensembles_day83.csv', index_col=0)
print('Day 83 LogReg v2 AUC:', day83.loc['LogReg v2', 'roc_auc'], '| tracked:', round(s1['cv_roc_auc_mean'], 4))
assert abs(s1['cv_roc_auc_mean'] - day83.loc['LogReg v2', 'roc_auc']) < 1e-3

# Wahi run dobara: metrics identical aane chahiye (reproducibility)
rid2, s2 = run_cv_experiment('logreg_v2_balanced_C1_rerun', model)
print('max metric diff between identical runs:', max(abs(s1[k] - s2[k]) for k in s1))

# Ek alag setting
rid3, s3 = run_cv_experiment('logreg_v2_balanced_C0.1', clone(model).set_params(m__C=0.1))
print('C=0.1 AUC:', round(s3['cv_roc_auc_mean'], 4))

NameError: name 'model' is not defined

Do checks yahan sabse zaroori hain. Pehla: tracking ne wahi number diya jo tumne Day 83 mein report kiya tha (agar fark aaye, to ya folds ya features alag hain, aur wo tracking se pehle ki bug hai). Doosra: same run dobara chalane par metrics same aate hain.

6: Runs ko code se query karo

In [17]:
runs = mlflow.search_runs(experiment_names=['churn-day91-intro'],
                          order_by=['metrics.cv_roc_auc_mean DESC'])
cols = ['tags.mlflow.runName', 'metrics.cv_roc_auc_mean', 'metrics.cv_roc_auc_std',
        'params.hp_C', 'tags.git_commit', 'tags.git_dirty']
print(runs[cols].round(4))

KeyError: "['metrics.cv_roc_auc_mean', 'metrics.cv_roc_auc_std', 'params.hp_C', 'tags.git_commit', 'tags.git_dirty'] not in index"

#### 5. UI tour (browser mein khud karo)

1. Runs table mein “Columns” se cv_roc_auc_mean, hp_C, git_dirty dikhao.
2. Search box mein try karo: tags.test_set_used = 'false' aur metrics.cv_roc_auc_mean > 0.5.
3. logreg_v2_balanced_C1 kholo: Parameters, Metrics (cv_fold_roc_auc ka step chart), Artifacts (fold_roc_auc.png, features.json), Tags.
4. Teeno runs select karke Compare dabao. C=1 aur C=0.1 ka fark parameters aur metrics dono mein dikhega.
5. git_dirty tag dekho. Agar True hai, to notebook ya code mein uncommitted badlav tha, aur commit hash poora sach nahi batata.

#### 6. Dhyan rakhne ki baatein

- MLflow tracking after-the-fact bhi ho sakti hai, par uska matlab “pehle se tay” nahi hota. Day 83-84 ke experiments pehle ho chuke hain. Kal unhe MLflow mein dobara log karenge. Wo sirf record ban jaayega. Final model ka faisla jo Day 84 mein commit hua, wahi sach rahega.
- Metric naam ka niyam: CV ke natije cv_ se shuru honge. Sklearn ke test_roc_auc jaise keys CV ke validation folds ke hain (held-out test set ke nahi), isliye naam badalna zaroori hai. Guard galat naam par error dega.
- Store backup: mlflow_store/ Git mein nahi jaayega, aur wo reproducible nahi hai (runs dobara chalane padenge). Is liye jo numbers README mein jaane hain, unhe reports/ ki CSV/MD mein bhi save karo.